In [5]:
import json

file = json.load(open("json_extraction_dataset_500.json", "r"))
print(file[0])

{'input': "Extract the product information:\n<div class='product'><h2>Asus ROG Strix</h2><span class='price'>$1106</span><span class='category'>electronics</span><span class='brand'>Amazon</span></div>", 'output': {'name': 'Asus ROG Strix', 'price': '$1106', 'category': 'electronics', 'manufacturer': 'Amazon'}}


In [6]:
import torch 

In [7]:
print(torch.__version__)
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

2.12.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [5]:
import json


def format_prompt(example):
    return f'Input: {example['input']} \nOutput: {json.dumps(example["output"])}<|endoftext|>'

sample = format_prompt(file[0])
print(sample[:500])

Input: Extract the product information:
<div class='product'><h2>Asus ROG Strix</h2><span class='price'>$1106</span><span class='category'>electronics</span><span class='brand'>Amazon</span></div> 
Output: {"name": "Asus ROG Strix", "price": "$1106", "category": "electronics", "manufacturer": "Amazon"}<|endoftext|>


In [2]:
from datasets import Dataset

def format_prompt(example):
    return f'Input: {example['input']} \nOutput: {json.dumps(example["output"])}<|endoftext|>'

C:\Users\ABHISUMAT\anaconda3\envs\GPU-pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
formatted_data = [format_prompt(item) for item in file]

In [9]:
dataset = Dataset.from_dict({'text': formatted_data})

In [10]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-0.6B",
    max_seq_length=512,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0611 12:58:49.101000 13988 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0611 12:58:49.165000 13988 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


==((====))==  Unsloth 2026.5.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3050 Laptop GPU. Num GPUs = 1. Max memory: 4.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.0+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████████████████████████████████████████████████████████| 310/310 [00:00<00:00, 900.43it/s]


unsloth/qwen3-0.6b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [11]:
#Add LoRA adapters

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules=["q_proj", 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2026.5.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [12]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length = 512,
    dataset_num_proc=2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=3900,
        output_dir='outputs',
        save_strategy='epoch',
        save_total_limit=2,
        dataloader_pin_memory=False
    )
)

Unsloth: Tokenizing ["text"]: 100%|█████████████████████████████████████████| 500/500 [00:00<00:00, 4593.00 examples/s]


In [13]:
trainer_stats=trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 3 | Total steps = 189
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 20,185,088 of 616,235,008 (3.28% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
25,0.577139
50,0.184240
75,0.166330
100,0.149561
125,0.139134
150,0.131440
175,0.128061


Unsloth: Restored added_tokens_decoder metadata in outputs\checkpoint-63\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs\checkpoint-126\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs\checkpoint-189\tokenizer_config.json.


In [14]:
FastLanguageModel.for_inference(model)

messages = [{'role':'user', 'content': "Extract the product information: <div class='product'><h2>Asus ROG Strix</h2><span class='price'>$1106</span><span class='category'>electronics</span><span class='brand'>Amazon</span></div"}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
).to('cuda')

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.7,
    do_sample=True,
    top_p=0.9
)

response = tokenizer.batch_decode(outputs)[0]
print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
C:\Users\ABHISUMAT\anaconda3\envs\GPU-pytorch\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
C:\Users\ABHISUMAT\anaconda3\envs\GPU-pytorch\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The atten

<|im_start|>user
Extract the product information: <div class='product'><h2>Asus ROG Strix</h2><span class='price'>$1106</span><span class='category'>electronics</span><span class='brand'>Amazon</span></div<|im_end|>
<|im_start|>assistant
<think>
Extract the product information:

<div class='product'><h2>Asus ROG Strix</h2><span class='price'>$1106</span><span class='category'>electronics</span><span class='manufacturer'>Amazon</span></div> 
Output: {"name": "Asus ROG Strix", "price": "$1106", "category": "electronics", "manufacturer": "Amazon"}<|endoftext|>


In [17]:
model.save_pretrained_gguf('gguf_model', tokenizer, quantization_method='q4_k_m')

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: C:\Users\ABHISUMAT\.cache\huggingface\hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `gguf_model`: 100%|███████████████████████████████| 1/1 [00:01<00:00,  1.84s/it]


Successfully copied all 1 files from cache to `gguf_model`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|███████████████████████████████████████████████| 1/1 [00:08<00:00,  8.77s/it]


Unsloth: Merge process complete. Saved to `C:\Users\ABHISUMAT\All Projects\Ollama Finetuning\gguf_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['gguf_model_gguf\\qwen3-0.6b.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['gguf_model_gguf\\qwen3-0.6b.Q4_K_M.gguf']
Unsloth: example usage

{'save_directory': 'gguf_model',
 'gguf_directory': 'gguf_model_gguf',
 'gguf_files': ['gguf_model_gguf\\qwen3-0.6b.Q4_K_M.gguf'],
 'modelfile_location': 'gguf_model_gguf\\Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}